In [1]:
import chromadb
import os
from dotenv import load_dotenv
from chromadb.utils import embedding_functions
from edgar import *
import yfinance as yf
from bs4 import BeautifulSoup
import requests
import re

Connect to the Chroma Vector DB

In [2]:
# Path to .env location containing the Chroma API keys
env_path = os.path.join("../data", ".env")

# Parse the .env file and retrieve the API keys
if load_dotenv(env_path):
    print("✅ Environment variables loaded from data/.env")
else:
    print("❌ Failed to load data/.env - Check if the file exists!")

CF_CLIENT_ID = os.getenv("CF-ACCESS-CLIENT-ID")
CF_CLIENT_SECRET = os.getenv("CF-ACCESS-CLIENT-SECRET")
CHROMA_URL = os.getenv("CHROMA_URL")

print(f"ID: {CF_CLIENT_ID}")
print(f"Secret: {CF_CLIENT_SECRET}")
print(f"Chroma URL: {CHROMA_URL}")


✅ Environment variables loaded from data/.env
ID: 15703ac0e5797dac885c851f20afe6cb.access
Secret: 9db6c201a502a20d0aa9eae4dac06d36b2f6a29b46201c20df2f5174263ddf57
Chroma URL: chroma.taskcomply.com


In [3]:
# 2. Setup the Client with Custom Headers
client = chromadb.HttpClient(
    host=CHROMA_URL,                              # Your Cloudflare URL
    port=443,                                     # Standard HTTPS port
    ssl=True,                                     # MUST be True for Cloudflare
    headers={
        "CF-Access-Client-Id": CF_CLIENT_ID,
        "CF-Access-Client-Secret": CF_CLIENT_SECRET
    },
)

# 3. Test the connection
print(f"Connection Heartbeat: {client.heartbeat()}")

Connection Heartbeat: 1775452960896669815


Determine embedding function and create 2 streams:
- Fundamentals
- Sentiment

In [4]:
# 4. Use a lightweight embedding function
# Default is 'all-MiniLM-L6-v2' which is great for financial headlines
emb_fn = embedding_functions.DefaultEmbeddingFunction()

# 3. Create the two streams
fundamentals_col = client.get_or_create_collection(
    name="stream1_fundamentals",
    embedding_function=emb_fn
)

sentiment_col = client.get_or_create_collection(
    name="stream2_sentiment",
    embedding_function=emb_fn
)

In [5]:
collections = client.list_collections()

print(collections)

[Collection(name=stream1_fundamentals), Collection(name=stream2_sentiment)]


1. Stream 1: Ingesting Fundamentals (The "Ground Truth")
We will use edgartools to pull the latest 10-Ks. For RAG to be effective, we don't just want the whole document; we want the Risk Factors (Item 1A), as these provide the best "analytical anchor."

In [6]:
# Set tickers to ingest
tickers = ["NVDA", "AAPL", "TSLA"]

# MUST set your identity for SEC compliance
set_identity("david.j.bain@student.uts.edu.au")

In [7]:
company = Company("NVDA")
# Get the latest Annual Report
filing = company.get_filings(form="10-K").latest()

# Extract the specific "Risk Factors" section as Markdown
# EdgarTools 2026 handles the section parsing automatically
tenk = filing.obj()

risk = tenk.risk_factors

print("Fundamentals Stream Hydrated.")

Fundamentals Stream Hydrated.


2. Stream 2: Ingesting Sentiment (The "Market Pulse")
For the sentiment stream, we want the most recent headlines. Since Tesla (TSLA) just had a delivery miss on April 2nd, and Apple (AAPL) just celebrated its 50th anniversary on April 1st, there is plenty of fresh data.

In [8]:
# Filter news article from 'noise' and pick up the primary tickers
# In yFinance we are searching for this example meta tag:
# <meta name="tickers" content="META,PATH,RELY"/>

def get_verified_article_data(url, target_ticker):
    try:
        headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64)'}
        response = requests.get(url, headers=headers, timeout=10)
        soup = BeautifulSoup(response.content, 'html.parser')

        # --- 1. VERIFICATION GATE ---
        # Using the tag: <meta name="primary_tickers" content="META"/>
        primary_meta = soup.find("meta", {"name": "primary_tickers"})
        if primary_meta:
            if target_ticker.upper() not in primary_meta.get("content", "").upper():
                return None

        # --- 2. CLEANING PHASE ---
        # We physically remove the 'ticker-bar' and 'image' divs so the scraper doesn't stop when it hits them.
        # Added 'fig-caption' to the noise list to avoid image descriptions
        for noise in soup.find_all(["div", "section", "header"],
                                  class_=["ticker-bar", "image", "ad-wrapper", "fig-caption"]):
            noise.decompose()

        # --- 3. EXTRACTION PHASE ---
        # Look for the container that holds the 'meat' of the article
        container = soup.find("div", class_=["article-body", "caas-body", "p-article-content"])

        if container:
            # We use a Regular Expression to find ALL Paragraphs AND Headings (h1, h2, h3, h4)
            # This ensures they stay in the order they appear on the screen
            text_elements = container.find_all(["p", re.compile('^h[1-4]$')])

            # Join them with a newline so sub-headings don't "smush" into paragraphs
            full_text = "\n\n".join([el.get_text().strip() for el in text_elements if len(el.get_text()) > 5])
            return full_text

        return None
    except Exception as e:
        print(f"Error on {url}: {e}")
        return None

In [9]:
ticker = "NVDA"

In [10]:
raw_news = yf.Ticker(ticker).news

print(raw_news[0])

{'id': 'dcdeb578-670c-3a4f-acea-c40ca6c23f42', 'content': {'id': 'dcdeb578-670c-3a4f-acea-c40ca6c23f42', 'contentType': 'STORY', 'title': '3 Growth Stocks Down 43%, 28%, and 41% to Buy Right Now', 'description': '', 'summary': "Don't become so fearful of recent weakness that you forget such pullbacks are usually great long-term buying opportunities.", 'pubDate': '2026-04-06T02:20:00Z', 'displayTime': '2026-04-06T02:20:00Z', 'isHosted': True, 'bypassModal': False, 'previewUrl': None, 'thumbnail': {'originalUrl': 'https://media.zenfs.com/en/motleyfool.com/52ae6f55b1883df19eca4f0df6524ee1', 'originalWidth': 1400, 'originalHeight': 933, 'caption': 'A person is taking care of some banking matters while seated in front of a laptop.', 'resolutions': [{'url': 'https://s.yimg.com/uu/api/res/1.2/92vIfyH.mS5Z02eG7jRDbw--~B/aD05MzM7dz0xNDAwO2FwcGlkPXl0YWNoeW9u/https://media.zenfs.com/en/motleyfool.com/52ae6f55b1883df19eca4f0df6524ee1', 'width': 1400, 'height': 933, 'tag': 'original'}, {'url': 'htt

In [11]:
docs = []
metas = []
ids = []

for i, item in enumerate(raw_news):
    content = item.get("content") or {}
    publisher = content.get("provider").get("displayName") or {}
    canonicalUrl = content.get("canonicalUrl").get("url") or {}

    clean_content = get_verified_article_data(canonicalUrl, ticker)

    if clean_content:
        docs.append(clean_content)

        metas.append({
            "ticker": ticker,
            "headline": content.get("title"),
            "date": content.get("pubDate"),
            "publisher": publisher,
            "url": canonicalUrl,
            "stream": "sentiment"
        })

        ids.append(f"{ticker}_news_{i}_{item.get('id')}")

print("Sentiment Stream Hydrated.\n")

print(metas[0]['headline'])
print(docs[0])
print(metas[0])
print(ids[0], "\n")

Sentiment Stream Hydrated.

5 Things to Know About OpenAI Before Its IPO
OpenAI quickly became the leading artificial intelligence (AI) company when ChatGPT was launched and jump-started the AI race a little over three years ago. The average investor hasn't been able to invest directly in the company, but that's likely to change soon, with OpenAI expected to have its initial public offering (IPO) later this year.

Even if you're a regular ChatGPT user, you might not know some of the details about the company. So, here are five things potential investors should know before the OpenAI IPO.

1. OpenAI could go public in Q4 2026 at a potential $1 trillion valuation

Multiple reports increasingly suggest its stock could debut as soon as the fourth quarter this year. That timeline isn't set in stone, of course, but the company has made other moves indicating it's getting closer to an IPO, such as expanding its finance team to include an employee dedicated to investor relations and reorientin